# 1. Curation and the functional API

Every QSAR model is bounded by the quality of the activity data behind it.
This notebook covers the first stage of the workflow: turning a pile of
structures and measurements into a dataset worth modelling.

Two equivalent notations are shown throughout — the scikit-learn estimator
API, and the left-to-right pipe API in `qsarkit.functional`. Use whichever
reads better; they call the same code.

**Covers:** `qsarkit.chemistry`, `qsarkit.data_quality`, `qsarkit.functional`,
`qsarkit.utils`

In [1]:
import numpy as np
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")   # we parse deliberately-bad input below
np.set_printoptions(precision=3, suppress=True)

## The dataset

A small congeneric-ish series: four substituent families with synthetic
pIC50 values. Deliberately included are a salt, a duplicate, an unparseable
string and an InChI, because real datasets contain all four.

In [2]:
SMILES = [
    "OC(=O)c1ccccc1", "OC(=O)c1ccc(C)cc1", "OC(=O)c1ccc(Cl)cc1",
    "OC(=O)c1ccc(Br)cc1", "OC(=O)c1ccc(OC)cc1", "OC(=O)c1ccc(N)cc1",
    "CC(=O)Nc1ccccc1", "CC(=O)Nc1ccc(C)cc1", "CC(=O)Nc1ccc(Cl)cc1",
    "CC(=O)Nc1ccc(F)cc1", "CC(=O)Nc1ccc(OC)cc1", "CC(=O)Nc1ccc(O)cc1",
    "NC(=O)c1ccncc1", "NC(=O)c1ccc(C)nc1", "NC(=O)c1ccc(Cl)nc1",
    "NC(=O)c1ccc(OC)nc1", "CNC(=O)c1ccncc1", "CCNC(=O)c1ccncc1",
    "c1ccc2[nH]cnc2c1", "Cc1ccc2[nH]cnc2c1", "Clc1ccc2[nH]cnc2c1",
    "COc1ccc2[nH]cnc2c1", "Cn1cnc2ccccc21", "CCn1cnc2ccccc21",
]
Y = np.array([
    5.10, 5.35, 7.80, 5.40, 5.05, 4.90,
    6.20, 6.45, 6.70, 6.55, 6.10, 6.05,
    7.10, 7.35, 7.55, 7.20, 7.05, 6.95,
    8.00, 8.25, 8.45, 8.10, 7.90, 7.85,
])

# The messy version: a sodium salt, a duplicate, a bad string, an InChI.
RAW = SMILES + [
    "OC(=O)c1ccccc1.[Na+]",                      # salt form of entry 0
    "OC(=O)c1ccccc1",                            # exact duplicate of entry 0
    "not-a-molecule",                            # unparseable
    "InChI=1S/C7H6O2/c8-7(9)6-4-2-1-3-5-6/h1-5H,(H,8,9)",  # benzoic acid
]
RAW_Y = np.concatenate([Y, [5.05, 5.15, 6.00, 5.10]])
len(RAW), len(RAW_Y)

(28, 28)

## Starting a pipeline

`molecules()` accepts RDKit molecules, SMILES and InChI in any mixture.
Anything unparseable becomes `None` rather than raising, so it keeps its
position — and therefore its alignment with `y` — until you decide what to
do with it.

In [3]:
from qsarkit.functional import molecules

ms = molecules(RAW, RAW_Y)
print(ms)
print("unparseable:", [i for i, m in enumerate(ms.mols) if m is None])
print("InChI parsed to:", ms.smiles[-1])

<MoleculeSet 28 molecules, y shape (28,), 1 invalid, 1 steps>
unparseable: [26]
InChI parsed to: O=C(O)c1ccccc1


That index alignment is the whole point. It is the bookkeeping that
hand-written curation scripts get subtly wrong: drop a molecule without its
label and every subsequent activity shifts by one — and the model still
trains, just on nonsense.

In [4]:
from qsarkit.functional import (
    desalt, drop_invalid, remove_duplicates, standardize)

mols, y = (
    molecules(RAW, RAW_Y)
    >> standardize()          # salts stripped, charges neutralized, tautomers canonical
    >> drop_invalid()         # the None goes, and its label with it
    >> remove_duplicates(agg="mean", max_spread=1.0)
)
print(f"{len(RAW)} records in -> {len(mols)} out")
print("labels still aligned:", len(mols) == len(y))

28 records in -> 24 out
labels still aligned: True


### Use `>>`, not `>`

Python parses `a > b > c` as the chained comparison `(a > b) and (b > c)`,
so a `>`-based pipe would evaluate the first stage, throw the result away,
and return a comparison of the last two. There is no way to intercept that
from `__gt__` — so rather than misbehave quietly, it raises.

In [5]:
try:
    molecules(["CCO"]) > desalt() > drop_invalid()
except TypeError as exc:
    print(exc)

'>' cannot be used as a pipe operator in Python: 'a > b > c' is parsed as the chained comparison '(a > b) and (b > c)', which silently throws away your data. Use '>>' instead:
    X, y = molecules(X, y) >> desalt() >> remove_duplicates()


## Provenance

Each set carries a log of what happened to it. This is what OECD
Principle 2 (an unambiguous algorithm) asks you to be able to produce.

In [6]:
ms = (
    molecules(RAW, RAW_Y)
    >> standardize()
    >> drop_invalid()
    >> remove_duplicates(agg="mean")
)
for entry in ms.history:
    print(" ", entry)

  molecules(n=28)
  standardize()
  drop_invalid()
  remove_duplicates(agg='mean')


## Reusable protocols

Steps compose with each other, so a curation protocol is defined once and
applied to train and test alike — which is the only way to be sure they
were treated identically.

In [7]:
curate = standardize() >> drop_invalid() >> remove_duplicates(max_spread=1.0)

train = molecules(SMILES[:12], Y[:12]) >> curate
test = molecules(SMILES[12:], Y[12:]) >> curate
print(len(train), len(test))
print(repr(curate))

12 12
<Pipeline standardize() >> drop_invalid() >> remove_duplicates(max_spread=1.0)>


## The same thing, imperatively

Every step is dual-mode: called without data it defers, called with data it
runs immediately. And the underlying classes are ordinary scikit-learn
transformers.

In [8]:
from qsarkit.chemistry import MolecularStandardizer

mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)[O-].[Na+]")
print("estimator API:", Chem.MolToSmiles(MolecularStandardizer().transform([mol])[0]))

immediate, _ = standardize([mol])
print("functional API:", Chem.MolToSmiles(immediate[0]))

estimator API: CC(=O)Oc1ccccc1C(=O)O
functional API: CC(=O)Oc1ccccc1C(=O)O


## Auditing the curation

`DataCurationPipeline` does the same work and hands back a report naming
every removal and the stage responsible.

In [9]:
from qsarkit.data_quality import DataCurationPipeline

# molecules() already parsed and aligned everything; reuse it rather than
# re-parsing, and drop the unparseable record first.
prepared = molecules(RAW, RAW_Y) >> drop_invalid()

mols_c, y_c, report = DataCurationPipeline().run(prepared.mols, prepared.y)
print(report.summary())

Curation: 27 -> 24 records (88.9% retained)
  standardize                27 -> 27     (0 removed)
  validate                   27 -> 27     (0 removed)
  deduplicate                27 -> 24     (3 removed)


## Diagnosing the activity column

The single most damaging silent error in QSAR is a mixed activity column.
Nothing in the numbers announces it, so check explicitly.

In [10]:
from qsarkit.data_quality import check_activity_units

for label, values in [
    ("p-scale (correct)", Y),
    ("raw nM", np.array([1.0, 10.0, 1000.0, 1e6])),
    ("raw molar", np.array([1e-9, 5e-8])),
]:
    r = check_activity_units(values, endpoint="IC50")
    print(f"{label:20} logarithmic={r['looks_logarithmic']!s:5} "
          f"warnings={len(r['warnings'])}")
    for w in r["warnings"]:
        print(" " * 22 + w)

p-scale (correct)    logarithmic=True  warnings=0
raw nM               logarithmic=False warnings=1
                      Values span 6.0 orders of magnitude, which suggests a raw concentration scale. Convert to pActivity (-log10 molar) before modeling.
raw molar            logarithmic=False warnings=1
                      All values are below 1 (max 5e-08), which suggests raw molar concentrations rather than a pActivity scale. Convert with qsarkit.utils.to_pactivity before modeling.


Converting to a p-scale makes the unit explicit and puts the values on the
scale every model here assumes — and makes the errors roughly normal, which
is what the regression metrics assume in turn.

In [11]:
from qsarkit.utils import to_pactivity

print("IC50 = 1 nM    -> pIC50", float(to_pactivity(1.0, unit="nM")))
print("IC50 = 1000 nM -> pIC50", float(to_pactivity(1000.0, unit="nM")))

IC50 = 1 nM    -> pIC50 9.0
IC50 = 1000 nM -> pIC50 6.0


## Structure-level checks

Records that are not usable molecules for QSAR: mixtures whose activity
belongs to no single structure, inorganics outside the applicability of
organic descriptors, fragments too small to carry signal.

In [12]:
from qsarkit.data_quality import StructureValidator

suspects = [Chem.MolFromSmiles(s) for s in
            ("CCO", "[Na+].[Cl-]", "O", "[NH3+]CC(=O)[O-]", "CC(=O)[O-]")]
names = ["ethanol", "sodium chloride", "water", "glycine", "acetate"]

for i, issue in enumerate(StructureValidator().validate(suspects)):
    print(f"{names[issue.index]:16} {issue.code:12} {issue.message}")

sodium chloride  mixture      Record has 2 disconnected components; activity cannot be attributed to one structure.
sodium chloride  inorganic    Contains non-organic elements: ['Na'].
sodium chloride  no_carbon    Contains no carbon.
sodium chloride  too_small    2 heavy atoms, below the minimum of 3.
water            no_carbon    Contains no carbon.
water            too_small    1 heavy atoms, below the minimum of 3.
acetate          charged      Molecule carries a net formal charge of -1; check that neutralization ran.


Note that glycine raises nothing. The charge check looks at *net* charge,
so a zwitterion — glycine, ciprofloxacin, any betaine — is not mistaken for
a record that escaped neutralization. Acetate, a genuine unbalanced ion,
is flagged.

## Chemistry-specific curation

Two operations that matter for particular dataset types and are easy to
forget.

In [13]:
from qsarkit.chemistry import FragmentRemover, GlycanDetector, GlycanRemover

# Natural products: the sugar dominates the fingerprint but carries no activity
q3g = Chem.MolFromSmiles(
    "OC[C@H]1O[C@@H](Oc2c(-c3ccc(O)c(O)c3)oc3cc(O)cc(O)c3c2=O)"
    "[C@H](O)[C@@H](O)[C@@H]1O")
print("sugars found:", GlycanDetector().detect(q3g)["num_sugar_residues"])
print("aglycone:    ", Chem.MolToSmiles(GlycanRemover().remove(q3g)["aglycone"]))

# Synthesis intermediates: a protecting group is an artefact, not a pharmacophore
boc = Chem.MolFromSmiles("CC(C)(C)OC(=O)NCc1ccccc1")
print("deprotected: ", Chem.MolToSmiles(FragmentRemover().transform([boc])[0]))

sugars found: 1
aglycone:     O=c1cc(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12
deprotected:  NCc1ccccc1


## What we have

A curated dataset, aligned labels, and a record of how we got there.
Notebook 2 turns it into features.

In [14]:
mols, y = molecules(SMILES, Y) >> curate
print(f"{len(mols)} molecules, {len(y)} labels, "
      f"pIC50 {y.min():.2f}-{y.max():.2f}")

24 molecules, 24 labels, pIC50 4.90-8.45
